In [5]:
#!/usr/bin/env python3
"""
upload_kaggle_dataset.py

Usage:
  python upload_kaggle_dataset.py --username YOUR_KAGGLE_USERNAME --public
  # (defaults point to your Delhi airshed path; change via --src)

Prereqs:
  pip install kaggle
  # Put kaggle.json in ~/.kaggle/ (or set KAGGLE_USERNAME / KAGGLE_KEY env vars)
"""
import os
import sys
import json
import shutil
import argparse
from kaggle.api.kaggle_api_extended import KaggleApi

def ensure_kaggle_auth():
    has_env = os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY")
    cred_path = os.path.expanduser("/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/Foundation-Models/kaggle.json")
    has_file = os.path.isfile(cred_path)
    if not (has_env or has_file):
        sys.exit(
            "❌ Kaggle credentials not found.\n"
            "   Create an API token on Kaggle → download kaggle.json → place at ~/.kaggle/kaggle.json\n"
            "   or set env vars KAGGLE_USERNAME and KAGGLE_KEY."
        )
    if has_file:
        os.makedirs(os.path.dirname(cred_path), exist_ok=True)
        try:
            os.chmod(cred_path, 0o600)
        except PermissionError:
            pass

def stage_copy(src: str, dst: str):
    if not os.path.isdir(src):
        sys.exit(f"❌ Source folder not found: {src}")
    if os.path.exists(dst):
        print(f"[i] Cleaning existing stage: {dst}")
        shutil.rmtree(dst)
    print(f"[i] Copying:\n  {src}\n→ {dst}")
    shutil.copytree(src, dst)

def write_metadata(stage_dir: str, username: str, slug: str,
                   title: str, subtitle: str, description: str,
                   license_name: str, keywords: list):
    meta = {
        "title": title,
        "id": f"{username}/{slug}",
        "licenses": [{"name": license_name}],
    }
    if subtitle:
        meta["subtitle"] = subtitle
    if description:
        meta["description"] = description
    if keywords:
        meta["keywords"] = keywords
    meta_path = os.path.join(stage_dir, "dataset-metadata.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    return meta_path

def dataset_exists(api: KaggleApi, owner: str, slug: str) -> bool:
    try:
        api.dataset_view(owner, slug)  # raises if not found
        return True
    except Exception:
        return False

def main():
    parser = argparse.ArgumentParser(
        description="Create or update a Kaggle Dataset from a local folder."
    )
    parser.add_argument(
        "--src",
        default="/home/rishabh.mondal/Brick-Kilns-project/ijcai_2025_kilns/data/processed_data/sentinel/delhi_airshed",
        help="Source folder to upload",
    )
    parser.add_argument("--username", required=True, help="Your Kaggle username")
    parser.add_argument(
        "--slug",
        default="delhi-airshed-sentinel2-brick-kilns-v1",
        help="Dataset slug (unique under your account)",
    )
    parser.add_argument(
        "--title",
        default="Delhi Airshed Sentinel-2 Brick Kiln Processed Data (v1)",
    )
    parser.add_argument(
        "--subtitle",
        default="Processed Sentinel-2 chips, labels & splits for the Delhi airshed",
    )
    parser.add_argument(
        "--description",
        default=(
            "Processed Sentinel-2 dataset for brick-kiln detection in the Delhi airshed. "
            "Includes folder structure, CRS/resolution, and split info. Only open Sentinel-2 derivatives."
        ),
    )
    parser.add_argument(
        "--license",
        default="CC-BY-4.0",
        help="Kaggle-supported license (e.g., CC0-1.0, CC-BY-4.0, CC-BY-NC-4.0, etc.)",
    )
    parser.add_argument(
        "--keywords",
        nargs="*",
        default=["sentinel-2", "remote sensing", "brick kilns", "delhi", "airshed", "object detection"],
    )
    parser.add_argument(
        "--public",
        action="store_true",
        help="Make dataset public on creation (you can also flip this later on the web UI)",
    )
    parser.add_argument(
        "--dir-mode",
        choices=["zip", "tar", "skip"],
        default="zip",
        help='How to handle subdirectories when uploading ("zip" is safest for nested trees)',
    )
    parser.add_argument(
        "--stage-root",
        default=os.path.expanduser("~/.cache/kaggle_stage"),
        help="Staging root (copies your data here so originals stay untouched)",
    )
    parser.add_argument(
        "--version-notes",
        default="Initial upload",
        help="Version notes when updating an existing dataset",
    )
    args = parser.parse_args()

    ensure_kaggle_auth()

    stage_dir = os.path.join(args.stage_root, args.slug)
    os.makedirs(args.stage_root, exist_ok=True)
    stage_copy(args.src, stage_dir)

    meta_path = write_metadata(
        stage_dir,
        args.username,
        args.slug,
        args.title,
        args.subtitle,
        args.description,
        args.license,
        args.keywords,
    )
    print(f"[i] Wrote metadata: {meta_path}")

    # Optional README.md for nicer Kaggle page
    readme_path = os.path.join(stage_dir, "README.md")
    if not os.path.exists(readme_path):
        with open(readme_path, "w", encoding="utf-8") as f:
            f.write(f"# {args.title}\n\n{args.description}\n")

    api = KaggleApi()
    api.authenticate()

    print("[i] Checking if dataset exists on Kaggle…")
    if dataset_exists(api, args.username, args.slug):
        print("[i] Found existing dataset — creating a new version")
        api.dataset_create_version(
            stage_dir,
            version_notes=args.version_notes,
            convert_to_csv=False,
            dir_mode=args.dir_mode,
            quiet=False,
        )
        url = f"https://www.kaggle.com/datasets/{args.username}/{args.slug}"
        print(f"✅ Updated dataset: {url}")
    else:
        print("[i] Creating a brand new dataset")
        api.dataset_create_new(
            stage_dir,
            public=args.public,
            quiet=False,
            convert_to_csv=False,
            dir_mode=args.dir_mode,
        )
        url = f"https://www.kaggle.com/datasets/{args.username}/{args.slug}"
        print(f"✅ Created dataset: {url}")
        if not args.public:
            print("ℹ️ It is currently PRIVATE. Visit the URL to switch to Public when ready.")

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--src SRC] --username USERNAME
                             [--slug SLUG] [--title TITLE]
                             [--subtitle SUBTITLE] [--description DESCRIPTION]
                             [--license LICENSE] [--keywords [KEYWORDS ...]]
                             [--public] [--dir-mode {zip,tar,skip}]
                             [--stage-root STAGE_ROOT]
                             [--version-notes VERSION_NOTES]
ipykernel_launcher.py: error: the following arguments are required: --username


SystemExit: 2